In [2]:
import pandas as pd
import altair as alt

# Basic set up
df = pd.read_csv('../data/Bank_Marketing_Dataset.csv')
df['Subscribed'] = df['TermDepositSubscribed'].map({1: 'Yes', 0: 'No'})

In [3]:
# Viz 4: Age vs Job Title Subscription Heatmap

# Bin age into decades
df['AgeBinLabel'] = pd.cut(
    df['Age'],
    bins=[20, 30, 40, 50, 60, 100],
    labels=['20s', '30s', '40s', '50s', '60s+'],
    right=False
)

# Pre-aggregate: subscription rate per age bin + job title
heatmap_data = (
    df.groupby(['AgeBinLabel', 'JobTitle'])['TermDepositSubscribed']
    .mean()
    .reset_index()
)
heatmap_data.columns = ['AgeBinLabel', 'JobTitle', 'SubscriptionRate']
heatmap_data['SubscriptionRate'] = (heatmap_data['SubscriptionRate'] * 100).round(1)

# Chart
chart_heatmap = alt.Chart(heatmap_data).mark_rect().encode(
    x=alt.X('AgeBinLabel:O', title='Age Group',
            sort=['20s', '30s', '40s', '50s', '60s+']),
    y=alt.Y('JobTitle:N', title='Job Title',
            sort=alt.EncodingSortField(field='SubscriptionRate', op='mean', order='descending')),
    color=alt.Color(
        'SubscriptionRate:Q',
        title='Subscription Rate (%)',
        scale=alt.Scale(scheme='blues')   # darker = higher subscription rate
    ),
    tooltip=[
        alt.Tooltip('AgeBinLabel:O', title='Age Group'),
        alt.Tooltip('JobTitle:N', title='Job Title'),
        alt.Tooltip('SubscriptionRate:Q', title='Subscription Rate (%)')
    ]
).properties(
    title='Subscription Rate by Age Group and Job Title',
    width=500,
    height=400
)

chart_heatmap.save('../visualizations/Chart4.html')
chart_heatmap

alt.Chart(...)